# 2-semantic-analysis-openai.ipynb

This notebook uses the OpenAI API to perform a semantic study on a dataset of news articles collected from Google Alerts.

We focus on rows from a Google Spreadsheet, specifically columns:
- G: Detected Language
- H: Detected Country
- I: Extracted Text

For each valid entry, the notebook uses OpenAI to:
1. Summarize the text.
2. Extract a "semantic universe" — a list of themes, concepts, and related areas.
3. Generate a list of relevant keywords.

This step builds on previously cleaned and language-processed data (see `0-clean-duplicates.ipynb` and `1-url-language-country-analyzer.ipynb`).

# Step 1: Install & import dependencies

In [1]:
!pip install --upgrade openai google-api-python-client google-auth google-auth-oauthlib python-dotenv pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.74.0
    Uninstalling openai-1.74.0:
      Successfully uninstalled openai-1.74.0


In [2]:
import os
import pandas as pd
import logging
from dotenv import load_dotenv
from googleapiclient.discovery import build
from google.oauth2 import service_account
from openai import OpenAI

# Step 2: Load credentials and connect to Google Sheets

In [3]:
# Load environment variables
load_dotenv()

SERVICE_ACCOUNT_FILE = os.getenv("SERVICE_ACCOUNT_FILE")
SPREADSHEET_ID = os.getenv("SPREADSHEET_ID")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) # You must add this to your .env file manually

# Define scope and authenticate with Google
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
sheet = build("sheets", "v4", credentials=creds).spreadsheets()

# Step 3: Load all rows from Sheet0 and filter by `wordAlert == "bioart"`

To avoid analyzing unrelated data, we only process rows where the `wordAlert` column contains `"bioart"`.  
This preserves alignment with the main dataset and allows us to update only the correct rows later.

In [4]:
# Read the full sheet (A to I to cover wordAlert to text)
result = sheet.values().get(
    spreadsheetId=SPREADSHEET_ID,
    range="Sheet0!A2:I"
).execute()

rows = result.get("values", [])

# Define column names (adjust as needed)
columns = [
    "wordAlert",  # A
    "link",       # B
    "date",       # C
    "source",     # D
    "title",      # E
    "description",# F
    "detected-language",  # G
    "detected-country",   # H
    "text"                # I
]

# Normalize rows to fixed length
normalized = [row + [""] * (len(columns) - len(row)) for row in rows]
df_all = pd.DataFrame(normalized, columns=columns)

# Filter only rows where wordAlert == "bioart"
bioart_mask = df_all["wordAlert"].str.strip().str.lower() == "bioart"
df_bioart = df_all[bioart_mask].copy()

# Store the original row indexes (to write back later)
bioart_indices_in_sheet = df_bioart.index.tolist()

# Separate rows with and without "Blocked" text
df_bioart_blocked = df_bioart[df_bioart["text"].str.strip().str.lower() == "blocked"].copy()
df_bioart_valid = df_bioart[df_bioart["text"].str.strip().str.lower() != "blocked"].copy()

# Keep only relevant columns for analysis
df_bioart_valid = df_bioart_valid[["detected-language", "detected-country", "text"]]
df_bioart_blocked = df_bioart_blocked[["detected-language", "detected-country", "text"]]  # still needed to align results

df_bioart_valid.head()

,detected-language,detected-country,text
35,hr,Unknown,Bio Awaking: Spoj umetnosti i nauke za održivu...
36,de,Unknown,Für Seehamer Röster ist Kaffee eine Lebenseins...
224,hr,Unknown,Šta se dešava kada umetnici uđu u naučne labor...
225,el,Unknown,Εγκαίνια της έκθεσης “Το Μεταλλαξιογόνο Μέλλον...
226,pt,Unknown,Editora Roncarati - ALERTAS ANVISA EM 24.05.20...


# Step 4: Analyze each row, define prompt and semantic analysis logic

We use the OpenAI API to:
- Complete missing `language` and `country` if marked as "unknown"
- Generate a short summary
- Suggest a semantic universe
- List relevant keywords

This step returns 5 new values per row, matching the columns:  
`language`, `country`, `summary`, `semantics`, and `keywords`


In [5]:
def analyze_with_openai_flexible(detected_lang, detected_country, text):
    if not text.strip():
        return {
            "language": "No Text",
            "country": "No Text",
            "summary": "No Summary",
            "semantics": "No Semantics",
            "keywords": "No Keywords"
        }

    MAX_TOKENS_TEXT = 10000
    text = text[:MAX_TOKENS_TEXT]

    lang_map = {
        "en": "English", "pt": "Portuguese", "de": "German",
        "fr": "French", "es": "Spanish", "it": "Italian",
        "zh": "Chinese", "ja": "Japanese"
    }

    lang_final = lang_map.get(detected_lang.lower().strip(), "") if detected_lang else ""
    ask_language = not lang_final

    country_final = detected_country.strip() if detected_country and detected_country.lower() != "unknown" else ""
    ask_country = not country_final

    # Prompt header
    prompt = f"""
You are a semantic analysis engine.

Analyze the following text and return a structured JSON with the following fields:
- "language": Detected language in English (e.g., "German", "Portuguese")
- "country": Country of origin or reference in English (e.g., "Germany", "Brazil")
- "summary": A concise 1–2 sentence summary in English
- "semantics": 2–4 high-level thematic or disciplinary domains (e.g., "critical design", "biofabrication", "data literacy")
- "keywords": A comma-separated list of 5–10 relevant keywords in English

Text:
{text}

Always return all answers in English, regardless of the original language of the text.
If language or country is already provided below, use them:
"""

    if not ask_language:
        prompt += f"\nLanguage: {lang_final}"
    if not ask_country:
        prompt += f"\nCountry: {country_final}"

    prompt += "\n\nRespond only with a valid JSON object."

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=600
        )

        import json
        raw = response.choices[0].message.content
        parsed = json.loads(raw)

        return {
            "language": parsed.get("language", lang_final or "Unknown"),
            "country": parsed.get("country", country_final or "Unknown"),
            "summary": parsed.get("summary", "Missing"),
            "semantics": parsed.get("semantics", "Missing"),
            "keywords": parsed.get("keywords", "Missing"),
        }

    except Exception as e:
        logging.warning(f"OpenAI failed: {e}")
        return {
            "language": lang_final or "Error",
            "country": country_final or "Error",
            "summary": "Error",
            "semantics": "Error",
            "keywords": "Error"
        }


# Step 5: Apply to dataset

In [6]:
# Apply row by row, passing all required arguments
results = df_bioart_valid.apply(
    lambda row: analyze_with_openai_flexible(row["detected-language"], row["detected-country"], row["text"]),
    axis=1
)

# Convert results to DataFrame with same index as df_bioart_valid
results_df = pd.DataFrame(results.tolist(), index=df_bioart_valid.index)

# Combine OpenAI results with valid entries
df_bioart_valid = pd.concat([df_bioart_valid, results_df], axis=1)

# Add placeholder results for "Blocked" rows
for col in ["language", "country", "summary", "semantics", "keywords"]:
    df_bioart_blocked[col] = "Blocked"

# Combine both valid and blocked, restoring original order
df_bioart_final = pd.concat([df_bioart_valid, df_bioart_blocked])
df_bioart_final = df_bioart_final.sort_index()  # ensures order matches Sheet0

df_bioart_final.head()


,detected-language,detected-country,text,language,country,summary,semantics,keywords
35,hr,Unknown,Bio Awaking: Spoj umetnosti i nauke za održivu...,Serbian,Serbia,Bio Awaking project aims to address ecological...,"[Bioart, Environmental sustainability, Art and...","Bio Awaking, bioart, ecological challenges, ar..."
36,de,Unknown,Für Seehamer Röster ist Kaffee eine Lebenseins...,German,Austria,Seehamer Röster sieht Kaffee als Lebenseinstel...,"[Kaffeekultur, Nachhaltigkeit, Gesundheit, Kaf...","Kaffee, Rösterei, Kaffeekultur, Nachhaltigkeit..."
224,hr,Unknown,Šta se dešava kada umetnici uđu u naučne labor...,Serbian,Serbia,Bio art challenges the world in an incredible ...,"[bio art, art and science fusion, environmenta...","bio art, art and science, nature, environmenta..."
225,el,Unknown,Εγκαίνια της έκθεσης “Το Μεταλλαξιογόνο Μέλλον...,Greek,Greece,The exhibition 'The Mutagenic Future' is inaug...,"[bioart, biotechnology, interactive installati...","exhibition, Ionian University, Ken Rinaldo, Ad..."
226,pt,Unknown,Editora Roncarati - ALERTAS ANVISA EM 24.05.20...,Portuguese,Brazil,Alert 4486 (Technovigilance) - Communication f...,"[Healthcare products, Quality control, Regulat...","BioMérieux, Vitek2 AST-N408, Technovigilance, ..."


# Step 6: Write results back to Google Sheets (e.g., columns J, K, L)

In [9]:
# Ensure all output fields are strings
for col in ["language", "country", "summary", "semantics", "keywords"]:
    df_bioart_final[col] = df_bioart_final[col].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else str(x)
    )

# Prepare values to write (columns J to N)
values_to_write = df_bioart_final[["language", "country", "summary", "semantics", "keywords"]].values.tolist()

# Calculate actual row numbers in the spreadsheet (Google Sheets starts at 1 and has a header row)
first_row = min(bioart_indices_in_sheet) + 2
last_row = max(bioart_indices_in_sheet) + 2

# Build full range with empty rows where needed (to align with Sheet0)
bioart_index_set = set(bioart_indices_in_sheet)
bioart_iter = iter(values_to_write)
full_update_rows = []

for i in range(first_row, last_row + 1):
    if i - 2 in bioart_index_set:
        full_update_rows.append(next(bioart_iter))
    else:
        full_update_rows.append(["", "", "", "", ""])  # placeholder for non-bioart lines

# Write data to columns J:N
sheet.values().update(
    spreadsheetId=SPREADSHEET_ID,
    range=f"Sheet0!J{first_row}:N{last_row}",
    valueInputOption="RAW",
    body={"values": full_update_rows}
).execute()

print(f"✅ Semantic results written to Sheet0, rows {first_row}–{last_row}, columns J:N.")


✅ Semantic results written to Sheet0, rows 37–4626, columns J:N.
